### 记忆治理策略：压缩非系统消息

In [10]:
from typing import Any

import os

from dotenv import load_dotenv
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    ToolMessage,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime

load_dotenv(override=True)

model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    model_kwargs={"reasoning_effort": "none"},
)

# 生成摘要用的模型（这里直接复用主模型）
summary_model = model


@before_model
def trim_message(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """压缩除系统消息以外的历史消息。

    策略：系统消息原样保留；其余消息保留最近 2 条，更早的合并成一条摘要。
    """
    messages = state["messages"]
    if len(messages) <= 3:
        return None

    systemmessages = []
    othermessages = []
    for message in messages:
        if isinstance(message, SystemMessage):
            systemmessages.append(message)
        else:
            othermessages.append(message)

    # 非系统消息不足 3 条时无需压缩
    if len(othermessages) <= 2:
        return None

    to_summarize = othermessages[:-2]
    kept = othermessages[-2:]

    history = "\n".join(f"{m.type}: {m.content}" for m in to_summarize)
    summary = summary_model.invoke(
        [
            HumanMessage(
                content=f"用中文简洁总结以下对话的要点，保留关键事实、用户偏好与待办：\n\n{history}"
            )
        ]
    ).content

    # 关键：用 REMOVE_ALL_MESSAGES 清空旧消息，再按顺序放回
    # [系统消息...] + [摘要] + [最近保留的消息...]
    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *systemmessages,
            HumanMessage(content=f"[历史摘要] {summary}"),
            *kept
        ]
    }


agent = create_agent(model=model, middleware=[trim_message], checkpointer=InMemorySaver())


/Users/lijixu/PycharmProjects/langchain_demo/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:516: UserWarning: Parameters {'reasoning_effort'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


In [11]:
config = {"configurable": {"thread_id": "trim-demo"}}

turns = [
    SystemMessage(content="你是一个贴心的助手，回答尽量简短。"),
    HumanMessage(content="我叫小明，是个后端工程师。"),
    HumanMessage(content="我最喜欢用 Python。"),
    HumanMessage(content="我养了一只叫豆豆的猫。"),
    HumanMessage(content="我最近在学 LangGraph。"),
    HumanMessage(content="你还记得我叫什么、做什么工作、猫叫什么吗？"),
]

for index, message in enumerate(turns):
    result = agent.invoke({"messages": [message]}, config=config)
    if index > 0:
        print("用户：", message.content)
        print("助手：", result["messages"][-1].content[:80])

final_messages = agent.get_state(config).values["messages"]
print("最终消息条数：", len(final_messages))
print("系统消息是否保留：", any(isinstance(m, SystemMessage) for m in final_messages))
print("首条非系统消息：", final_messages[1].content)


用户： 我叫小明，是个后端工程师。
助手： 你好小明，后端工程师。有什么我可以帮你的吗？
用户： 我最喜欢用 Python。
助手： Python 做后端很合适。有什么具体想聊的吗？
用户： 我养了一只叫豆豆的猫。
助手： 豆豆这名字挺可爱。平时调皮吗？
用户： 我最近在学 LangGraph。
助手： LangGraph 挺适合做多智能体编排的。你是想搭 Agent 还是做 RAG 流程？
用户： 你还记得我叫什么、做什么工作、猫叫什么吗？
助手： 记得：

- 你叫小明
- 后端工程师，最爱用 Python
- 猫叫豆豆
最终消息条数： 5
系统消息是否保留： True
首条非系统消息： [历史摘要] - 小明是后端工程师，最爱用 Python。
- 小明养了一只叫豆豆的猫，AI 觉得名字可爱并问平时是否调皮。
- 小明最近在学 LangGraph。
- 暂无明确待办事项。


### 消息删除策略

上一节的「压缩」是把旧消息交给模型总结成摘要，**保留语义**；「删除」则是直接把消息丢弃，**不生成摘要、零额外成本**。

删除靠 `RemoveMessage` 实现，它是 `add_messages` reducer 支持的特殊指令：

- `RemoveMessage(id=某条消息的 id)`：删除指定的那一条消息；
- `RemoveMessage(id=REMOVE_ALL_MESSAGES)`：清空全部消息（配合重新放回实现『替换』）。

在 `@before_model` 等钩子里返回 `{"messages": [RemoveMessage(...), ...]}` 即可在模型调用前删消息。

| 策略 | 做法 | 适用场景 |
| --- | --- | --- |
| 滑动窗口 | 只保留最近 N 条 | 对话流，丢弃久远历史 |
| Token 裁剪 | 按 token 上限裁剪 | 精确控制上下文长度 |
| 精确删除 | 按 id 删除某条 | 删掉过期的工具结果/敏感消息 |

#### 策略 1：滑动窗口（保留最近 N 条）

系统消息始终保留，其余只保留最近 `keep_last` 条，更早的返回 `RemoveMessage` 删除。

In [12]:
from langchain.agents.middleware import after_model
from langchain_core.messages.utils import count_tokens_approximately, trim_messages


@after_model
def sliding_window(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """滑动窗口：系统消息始终保留，其余只留最近 4 条，更早的直接删除。"""
    messages = state["messages"]
    keep_last = 4
    if len(messages) <= keep_last:
        return None

    systemmessages = [m for m in messages if isinstance(m, SystemMessage)]
    othermessages = [m for m in messages if not isinstance(m, SystemMessage)]
    kept = systemmessages + othermessages[-keep_last:]
    kept_ids = {m.id for m in kept}

    # 只返回要删除的消息，add_messages 会按 id 把它们移除
    to_delete = [RemoveMessage(id=m.id) for m in messages if m.id not in kept_ids]
    print(f"  [滑动窗口] 删除 {len(to_delete)} 条，保留 {len(kept)} 条")
    return {"messages": to_delete}


window_agent = create_agent(
    model=model, middleware=[sliding_window], checkpointer=InMemorySaver()
)


In [13]:
window_config = {"configurable": {"thread_id": "window-demo"}}

for i in range(6):
    window_agent.invoke(
        {"messages": [{"role": "user", "content": f"这是第 {i} 句话。"}]}, config=window_config
    )

window_messages = window_agent.get_state(window_config).values["messages"]
print("滑动窗口后消息数：", len(window_messages))


  [滑动窗口] 删除 2 条，保留 4 条
  [滑动窗口] 删除 2 条，保留 4 条
  [滑动窗口] 删除 2 条，保留 4 条
  [滑动窗口] 删除 2 条，保留 4 条
滑动窗口后消息数： 4


#### 策略 2：按 Token 上限裁剪

用 `trim_messages` 按 token 数裁剪上下文（`include_system=True` 保证系统消息不被裁掉），再把被丢弃的消息交给 `RemoveMessage` 删除。

In [14]:
MAX_TOKENS = 60


@before_model
def token_trim(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """当估算 token 超过上限时，只保留最近的若干条。"""
    messages = state["messages"]
    if count_tokens_approximately(messages) <= MAX_TOKENS:
        return None

    trimmed = trim_messages(
        messages,
        max_tokens=MAX_TOKENS,
        token_counter=count_tokens_approximately,
        strategy="last",
        include_system=True,
        allow_partial=False,
    )

    # 清空旧消息后放回裁剪结果：remove-all + 重建，比算差集更简洁、也更鲁棒
    print(f"  [Token 裁剪] 保留 {len(trimmed)} 条，其余删除")
    return {"messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES), *trimmed]}


token_agent = create_agent(model=model, middleware=[token_trim], checkpointer=InMemorySaver())


In [15]:
token_config = {"configurable": {"thread_id": "token-demo"}}

token_agent.invoke(
    {
        "messages": [
            SystemMessage(content="回答尽量简短。"),
            HumanMessage(content="请记住：我叫小明，是后端工程师，喜欢 Python，养猫叫豆豆。"),
        ]
    },
    config=token_config,
)

for i in range(6):
    token_agent.invoke(
        {"messages": [{"role": "user", "content": f"再随便聊点别的第 {i} 句，用来填充上下文。"}]},
        config=token_config,
    )

print("Token 裁剪后消息数：", len(token_agent.get_state(token_config).values["messages"]))


  [Token 裁剪] 保留 5 条，其余删除
  [Token 裁剪] 保留 5 条，其余删除
  [Token 裁剪] 保留 5 条，其余删除
  [Token 裁剪] 保留 6 条，其余删除
  [Token 裁剪] 保留 6 条，其余删除
Token 裁剪后消息数： 7


#### 策略 3：精确删除指定消息

知道消息 `id` 时，可以直接调用 `agent.update_state(...)` 应用一条 `RemoveMessage`，**不会触发模型调用**，适合删掉过期的工具结果或敏感消息。

In [16]:
delete_agent = create_agent(model=model, checkpointer=InMemorySaver())
delete_config = {"configurable": {"thread_id": "delete-demo"}}

delete_agent.invoke(
    {"messages": [HumanMessage("第一句"), HumanMessage("第二句"), HumanMessage("第三句")]},
    config=delete_config,
)

before = delete_agent.get_state(delete_config).values["messages"]
print("删除前：", [m.content[:12] for m in before])

# 删除中间那条（"第二句"）
target = before[1]
delete_agent.update_state(delete_config, {"messages": [RemoveMessage(id=target.id)]})

after = delete_agent.get_state(delete_config).values["messages"]
print("删除后：", [m.content[:12] for m in after])


删除前： ['第一句', '第二句', '第三句', '可以，但你现在只给了“第']
删除后： ['第一句', '第三句', '可以，但你现在只给了“第']


#### 删除 vs 压缩：怎么选？

| | 压缩（摘要） | 删除 |
| --- | --- | --- |
| 额外成本 | 需要一次 LLM 调用 | 无 |
| 信息 | 保留语义要点 | 直接丢失 |
| 上下文长度 | 压缩但仍有摘要 | 真正变小 |
| 典型用法 | 长对话保留记忆 | 丢弃过期/冗余/敏感消息 |

实务中常**组合使用**：先用滑动窗口/token 裁剪控制长度，再把要丢弃的历史压缩成一段摘要，兼顾成本与记忆。

**要点**

1. `RemoveMessage(id=...)` 删单条，`RemoveMessage(id=REMOVE_ALL_MESSAGES)` 清空全部；
2. 删除是「返回要删的消息」，由 `add_messages` reducer 执行，不需要手动重建列表；
3. `trim_messages(..., include_system=True)` 可保证系统消息不被裁掉；
4. `agent.update_state(config, {...})` 不跑模型，适合精确删改记忆。